In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

The codes below are an edit of https://www.kaggle.com/code/xiaocao123/ai-generated-text-detection-quick-baselin-sgd

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
import xgboost as xgb
import lightgbm as lgb
from sklearn.linear_model import SGDClassifier
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
external_df = pd.read_csv("/kaggle/input/daigt-external-dataset/daigt_external_dataset.csv", sep=',')
external_df.head()

In [ ]:
#external_df['text'][0]

In [ ]:
#external_df['instructions'][0]

In [ ]:
#external_df['source_text'][0]

In [ ]:
external_df= external_df.rename(columns= {'generated': 'label'})
external_df.head()

In [ ]:
external_df= external_df[['source_text']]
external_df.head()

In [ ]:
external_df.columns= ["text"]
#external_df.head()

In [ ]:
external_df['text']= external_df['text'].str.replace('\n', '')
#external_df.head()
external_df['label']= 1
external_df.head()

In [ ]:
train = pd.read_csv("/kaggle/input/llm-7-prompt-training-dataset/train_essays_RDizzl3_seven_v1.csv")
train= pd.concat([train, external_df])
test = pd.read_csv('/kaggle/input/llm-detect-ai-generated-text/test_essays.csv')
df = pd.concat([train['text'], test['text']], axis=0)

df.head()

In [ ]:
vectorizer= TfidfVectorizer(stop_words= 'english', max_features= 25000, ngram_range= (1,3), sublinear_tf= True)
X= vectorizer.fit_transform(df)

In [ ]:
train.value_counts("label")

In [ ]:
RFC= RandomForestClassifier()
light_model= lgb.LGBMClassifier(learning_rate= 0.09, max_depth= -5, random_state= 42)
xgb_model= xgb.XGBClassifier()
SGD_model= SGDClassifier(max_iter= 1000, tol= 1e-3, loss= 'modified_huber')

In [ ]:
ensemble= VotingClassifier(estimators= [('Rf', RFC), ('light', light_model), ('xgb', xgb_model), ('sgd', SGD_model)], voting= 'soft')
ensemble.fit(X[:train.shape[0]], train.label)
# Create the ensemble model

In [ ]:
preds_test= ensemble.predict_proba(X[train.shape[0]:],)[:,1]

In [ ]:
pd.DataFrame({'id': test['id'], 'generated': preds_test}).to_csv('submission.csv', index= False)